In [ ]:
import os
import csv
import logging
import rdkit
import subprocess
import glob
import logging
import concurrent.futures
from dataclasses import dataclass
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.MolStandardize import rdMolStandardize




## Chem toolbox: 2d -> 3d

In [ ]:
# ==============================================================================
# Paths
# ==============================================================================

CSV_PATH  = Path("./top30_xg.csv")
OUT_DIR   = Path("./out_dir")
SMILES_COL = "SMILES"
PH         = 7.4

# ==============================================================================
# Logging
# ==============================================================================

logger = logging.getLogger("ligand_prep")
logger.setLevel(logging.INFO)
logger.handlers.clear()

fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
sh  = logging.StreamHandler()
sh.setFormatter(fmt)
fh  = logging.FileHandler(OUT_DIR.parent / "ligand_prep.log")
fh.setFormatter(fmt)
logger.addHandler(sh)
logger.addHandler(fh)

# ==============================================================================
# Pipeline functions
# ==============================================================================

def standardize(mol: Chem.Mol) -> Chem.Mol:
    Chem.SanitizeMol(mol)
    mol = rdMolStandardize.Cleanup(mol)
    mol = rdMolStandardize.FragmentParent(mol)
    mol = rdMolStandardize.Uncharger().uncharge(mol)
    mol = rdMolStandardize.TautomerEnumerator().Canonicalize(mol)
    return mol

def protonate(mol: Chem.Mol, ph: float = PH) -> Chem.Mol:
    smi = Chem.MolToSmiles(mol)
    result = subprocess.run(
        [
            "dimorphite_dl",
            "--ph_min", str(ph),
            "--ph_max", str(ph),
            "--precision", "0.5",
            smi,
        ],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"dimorphite_dl failed: {result.stderr.strip()}")
    lines = [l.strip() for l in result.stdout.strip().splitlines() if l.strip()]
    if not lines:
        raise ValueError(f"dimorphite_dl returned no results for: {smi}")
    return Chem.MolFromSmiles(lines[0])

def embed_and_minimize(mol: Chem.Mol) -> Chem.Mol:
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    if AllChem.EmbedMolecule(mol, params) == -1:
        raise RuntimeError("ETKDGv3 embedding failed")
    if AllChem.MMFFOptimizeMolecule(mol, mmffVariant="MMFF94s") != 0:
        AllChem.UFFOptimizeMolecule(mol)
    return mol


def write_sdf(mol: Chem.Mol, path: Path) -> None:
    writer = Chem.SDWriter(str(path))
    writer.write(mol)
    writer.close()




def sdf_to_pdbqt(sdf_path: Path, pdbqt_path: Path) -> None:
    result = subprocess.run(
        [
            "obabel",
            str(sdf_path),
            "-O", str(pdbqt_path),
            "--partialcharge", "gasteiger",
        ],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"obabel failed: {result.stderr.strip()}")


def prepare_ligand(smiles: str, out_pdbqt: Path, ph: float = PH) -> Path:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")
    mol      = standardize(mol)
    mol      = protonate(mol, ph=ph)
    mol      = embed_and_minimize(mol)
    sdf_path = out_pdbqt.with_suffix(".sdf")
    write_sdf(mol, sdf_path)
    sdf_to_pdbqt(sdf_path, out_pdbqt)
    sdf_path.unlink()
    return out_pdbqt

# ==============================================================================
# Run
# ==============================================================================

OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(CSV_PATH, newline="") as f:
    reader = csv.DictReader(f)

    if SMILES_COL not in reader.fieldnames:
        raise ValueError(
            f"Column '{SMILES_COL}' not found. "
            f"Available columns: {reader.fieldnames}"
        )

    counts = {"ok": 0, "skip": 0}

    for i, row in enumerate(reader):
        smiles   = row[SMILES_COL].strip()
        out_path = OUT_DIR / f"mol_{i:04d}.pdbqt"

        if out_path.exists():
            logger.info(f"[{i:04d}] SKIP (exists)  {smiles}")
            counts["skip"] += 1
            continue

        try:
            prepare_ligand(smiles, out_path, ph=PH)
            logger.info(f"[{i:04d}] OK    {smiles}")
            counts["ok"] += 1
        except Exception as e:
            logger.warning(f"[{i:04d}] FAIL  {smiles} — {e}")
            counts["skip"] += 1

logger.info(f"--- Done — OK: {counts['ok']}  Skipped/Failed: {counts['skip']} ---")

## Docking

In [ ]:

BASE_DIR        = Path("/home/mark/Documents/hack")
RECEPTOR_FILE   = BASE_DIR / "./2receptor_ready.pdbqt"
LIGAND_DIR      = BASE_DIR / "./out_dir"
OUTPUT_DIR      = BASE_DIR / "./out_dock"
SMINA_EXE       = BASE_DIR / "./smina.static"
LOG_FILE        = BASE_DIR / "docking.log"


In [ ]:
@dataclass
class DockingConfig:
    receptor_file:    Path = RECEPTOR_FILE
    ligand_dir:       Path = LIGAND_DIR
    output_dir:       Path = OUTPUT_DIR
    smina_executable: Path = SMINA_EXE
    log_file:         Path = LOG_FILE

    # Box parameters
    center_x: float = 30
    center_y: float = 0
    center_z: float = 7
    size_x:   float = 54
    size_y:   float = 54
    size_z:   float = 54

    # Smina parameters
    num_modes:      int = 9
    exhaustiveness: int = 9

    # Parallelization
    # None = auto: os.cpu_count() // cpu_per_task
    max_workers:  int = None
    cpu_per_task: int = 8


In [ ]:
logger = logging.getLogger("docking")
logger.setLevel(logging.INFO)
logger.handlers.clear()

fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
sh  = logging.StreamHandler()
sh.setFormatter(fmt)
fh  = logging.FileHandler(LOG_FILE)
fh.setFormatter(fmt)
logger.addHandler(sh)
logger.addHandler(fh)


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ligand_files = sorted(LIGAND_DIR.glob("*.pdbqt"))
if not ligand_files:
    raise FileNotFoundError(f"No .pdbqt ligands found in {LIGAND_DIR}")

pending = [
    f for f in ligand_files
    if not (OUTPUT_DIR / f"{f.stem}_docked.pdbqt").exists()
]
skipped = len(ligand_files) - len(pending)

logger.info(f"Ligands total:   {len(ligand_files)}")
logger.info(f"Already docked:  {skipped}")
logger.info(f"To process:      {len(pending)}")


In [ ]:
config = DockingConfig()

CENTER_X       = config.center_x
CENTER_Y       = config.center_y
CENTER_Z       = config.center_z
SIZE_X         = config.size_x
SIZE_Y         = config.size_y
SIZE_Z         = config.size_z
NUM_MODES      = config.num_modes
EXHAUSTIVENESS = config.exhaustiveness
CPU_PER_TASK   = config.cpu_per_task
MAX_WORKERS    = config.max_workers or (os.cpu_count() // config.cpu_per_task)

In [ ]:
def run_smina(ligand_file: str) -> dict:
    ligand_path = Path(ligand_file)
    ligand_name = ligand_path.stem
    output_file = OUTPUT_DIR / f"{ligand_name}_docked.pdbqt"
    task_log    = OUTPUT_DIR / f"{ligand_name}.log"

    if output_file.exists():
        return {"status": "skipped", "name": ligand_name, "output": str(output_file)}

    command = [
        str(SMINA_EXE),
        "--receptor",       str(RECEPTOR_FILE),
        "--ligand",         str(ligand_path),
        "--center_x",       str(CENTER_X),
        "--center_y",       str(CENTER_Y),
        "--center_z",       str(CENTER_Z),
        "--size_x",         str(SIZE_X),
        "--size_y",         str(SIZE_Y),
        "--size_z",         str(SIZE_Z),
        "--num_modes",      str(NUM_MODES),
        "--exhaustiveness", str(EXHAUSTIVENESS),
        "--out",            str(output_file),
        "--cpu",            str(CPU_PER_TASK),
        "--log",            str(task_log),
    ]

    try:
        subprocess.run(command, capture_output=True, text=True, check=True, timeout=900)
        return {"status": "success", "name": ligand_name, "output": str(output_file)}

    except subprocess.TimeoutExpired:
        return {"status": "timeout", "name": ligand_name, "output": None}

    except subprocess.CalledProcessError as e:
        task_log.write_text(e.stderr)
        return {"status": "failed", "name": ligand_name, "output": None, "error": e.stderr[:200]}

    except FileNotFoundError:
        raise RuntimeError(f"smina not found: {SMINA_EXE}")

In [ ]:
counts = {"success": 0, "failed": 0, "timeout": 0, "skipped": skipped}

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(run_smina, str(f)): f for f in pending}

    for i, future in enumerate(concurrent.futures.as_completed(futures), 1):
        try:
            result = future.result()
            counts[result["status"]] = counts.get(result["status"], 0) + 1
            logger.info(f"[{i}/{len(pending)}] {result['status'].upper():8s} {result['name']}")
        except RuntimeError as e:
            logger.critical(str(e))
            executor.shutdown(wait=False, cancel_futures=True)
            raise

logger.info("--- Done ---")
logger.info(f"  Success: {counts['success']}")
logger.info(f"  Skipped: {counts['skipped']}")
logger.info(f"  Failed:  {counts['failed']}")
logger.info(f"  Timeout: {counts['timeout']}")

In [ ]:
import glob
import re
import csv
import os
import pandas as pd
# Найти все .log файлы
log_files = glob.glob("./out_dock/mol_*.log")

results = []

for filepath in sorted(log_files):
    mol_name = os.path.splitext(os.path.basename(filepath))[0]  # mol_XXXX
    
    with open(filepath, "r") as f:
        content = f.read()
    
    # Найти первую строку с результатами (mode 1 — лучший результат)
    match = re.search(r"^\s*1\s+([-\d.]+)", content, re.MULTILINE)
    
    if match:
        affinity = float(match.group(1))
        results.append((mol_name, affinity))
    else:
        results.append((mol_name, None))  # если не распарсилось

# Сохранить в CSV
with open("resultsincept.csv", "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["molecule", "affinity_kcal_mol"])
    writer.writerows(results)

input_df = pd.read_csv("./top30_pls_incept.csv", sep=",")
output_df = pd.read_csv("resultsincept.csv")
output_df["idx"] = output_df["molecule"].str.extract(r"mol_(\d+)").astype(int)
output_df["SMILES"] = output_df["idx"].map(input_df["SMILES"])
output_df = output_df.drop(columns=["idx"])
output_df.to_csv("results_x.csv", index=False)

print(f"Обработано файлов: {len(results)}")
print("Сохранено в results.csv")

In [ ]:
import pandas as pd
import re

input_df = pd.read_csv("./top30_xg.csv")
output_df = pd.read_csv("results_x.csv")

# Индекс из имени файла (mol_0 -> 0, mol_1 -> 1)
output_df["idx"] = output_df["molecule"].str.extract(r"mol_(\d+)").astype(int)

# Маппинг по позиционному индексу DataFrame (iloc-style)
output_df["SMILES"] = output_df["idx"].apply(lambda i: input_df.iloc[i]["SMILES"])

output_df = output_df.drop(columns=["idx"])
print(output_df[["molecule", "affinity_kcal_mol", "SMILES"]])
output_df.to_csv("results.csv", index=False)